<center><img src="../../logo.png" alt="Header" style="width: 800px;"/></center>


@Copyright (C): 2010-2020, Shenzhen Yahboom Tech  
@Author: Liusen  
@Date: 2020-02-09 17:20:02  
@LastEditors: xiaozhen
@LastEditTime: 2020-09-30 18:20:02   

In [1]:
#bgr8转jpeg格式
import enum
import cv2
import os

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

In [2]:
#摄像头显示组件
import cv2
import ipywidgets.widgets as widgets
import threading
import time
import sys
import libcamera

from picamera2 import Picamera2




In [3]:
# 线程功能操作库
import inspect
import ctypes
def _async_raise(tid, exctype):
    """raises the exception, performs cleanup if needed"""
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        # """if it returns a number greater than one, you're in trouble,
        # and you should call it again with exc=NULL to revert the effect"""
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

@ 人脸检测器（默认）：haarcascade_frontalface_default.xml  

@ 人脸检测器（快速Harr）：haarcascade_frontalface_alt2.xml   

@ 人脸检测器（侧视）：haarcascade_profileface.xml   

@ 眼部检测器（左眼）：haarcascade_lefteye_2splits.xml  

@ 眼部检测器（右眼）：haarcascade_righteye_2splits.xml   

@ 嘴部检测器：haarcascade_mcs_mouth.xml  

@ 鼻子检测器：haarcascade_mcs_nose.xml  

@ 身体检测器：haarcascade_fullbody.xml  

@ 人脸检测器（快速LBP）：lbpcascade_frontalface.xml  

@ 仅可以检测睁开的眼睛：  
@ haarcascade_eye.xml  
@ haarcascade_eye_tree_eyeglasses.xml [仅在带被检测者戴眼镜时方可检测]  
@ 用户可以进入这个网址https://github.com/opencv/opencv/tree/master/data  下载分类器文件，然后将下载的文件放置到这个程序所在的目录下面。

@ 注意以下返回值都是四个：坐标X，坐标Y，宽W，高H  


In [4]:
# body_haar = cv2.CascadeClassifier("haarcascade_upperbody.xml")
face_haar = cv2.CascadeClassifier("haarcascade_frontalface_default.xml") #检测人脸
eye_haar = cv2.CascadeClassifier("haarcascade_eye_tree_eyeglasses.xml")  #检测戴眼镜的眼睛
def Camera_display():
    while 1:
        frame = picam2.capture_array()
        # 把图像转为黑白图像
        gray_img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#         # 检测图像中的所有行人
#         bodies = body_haar.detectMultiScale(gray_img, 1.3, 5)
#         for body_x,body_y,body_w,body_h in bodies:
#             cv2.rectangle(frame, (body_x, body_y), (body_x+body_w, body_y+body_h), (0,255,0), 2)

# detectMultiScale（const Mat& image, vector& objects, double scaleFactor=1.1，int minNeighbors, int flag，cvSize）
# 1. image为输入的灰度图像
# 2. objects为得到被检测物体的矩形框向量组
# 3. scaleFactor为每一个图像尺度中的尺度参数，默认值为1.1。scale_factor参数可以决定两个不同大小的窗口扫描之间有多大的跳跃，这个参数设置的大，则意味着计算会变快，但如果窗口错过了某个大小的人脸，则可能丢失物体。
# 4. minNeighbors参数为每一个级联矩形应该保留的邻近个数（没能理解这个参数，-_-|||），默认为3。minNeighbors控制着误检测，默认值为3表明至少有3次重叠检测，我们才认为人脸确实存。
# 5. minSize为目标的最小尺寸
# 6. minSize为目标的最大尺寸


        faces = face_haar.detectMultiScale(gray_img, 1.1, 3)
        for face_x,face_y,face_w,face_h in faces:
            cv2.rectangle(frame, (face_x, face_y), (face_x+face_w, face_y+face_h), (0,255,0), 2)   #绿色线框框出检测到的人脸
            
        eyes = eye_haar.detectMultiScale(gray_img, 1.1, 3)
        for eye_x,eye_y,eye_w,eye_h in eyes:
            cv2.rectangle(frame, (eye_x,eye_y), (eye_x+eye_w, eye_y+eye_h), (255,0,0), 2)    #蓝色线框框出检测到的戴眼镜的眼睛
            
        image_widget.value = bgr8_to_jpeg(frame)
        time.sleep(0.010)

In [5]:
image_widget = widgets.Image(format='jpeg', width=320, height=240)
display(image_widget)

Image(value=b'', format='jpeg', height='240', width='320')

In [6]:
cv2.startWindowThread()
picam2 = Picamera2()
config = picam2.create_preview_configuration(main={"format": 'RGB888', "size": (640, 480)})
#config["transform"] = libcamera.Transform(hflip=0, vflip=1)
picam2.configure(config)
picam2.start()

frame = picam2.capture_array()
image_widget.value = bgr8_to_jpeg(frame)


[0:17:09.699544606] [11899]  INFO Camera camera_manager.cpp:313 libcamera v0.3.0+65-6ddd79b5
[0:17:09.712614525] [11930]  INFO RPI pisp.cpp:695 libpisp version v1.0.6 b567f0455680 17-06-2024 (10:20:00)
[0:17:09.732721655] [11930]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@80000/imx219@10 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:17:09.734324294] [11899]  INFO Camera camera_manager.cpp:313 libcamera v0.3.0+65-6ddd79b5
[0:17:09.746250603] [11933]  INFO RPI pisp.cpp:695 libpisp version v1.0.6 b567f0455680 17-06-2024 (10:20:00)
[0:17:09.761029289] [11933]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@80000/imx219@10 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:17:09.765663646] [11899]  WARN V4L2 v4l2_pixelformat.cpp:344 Unsupported V4L2 pixel format RPBP
[0:17:09.766541041] [11899]  INFO Camera camera.cpp:1183 configuring streams: (0) 640x480-RGB888 (1

In [7]:
#启动线程
thread1 = threading.Thread(target=Camera_display)
thread1.setDaemon(True)
thread1.start()

/tmp/ipykernel_11899/1187651759.py:3: DeprecationWarning: setDaemon() is deprecated, set the daemon attribute instead
  thread1.setDaemon(True)


In [9]:
#结束进程，释放摄像头，需要结束时执行
picam2.stop()
stop_thread(thread1)

ValueError: invalid thread id